In [ ]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/train.csv", parse_dates=["Date"])
store = pd.read_csv("data/store.csv")

df = train.merge(store, on="Store", how="left")

df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df["Month"] = df["Date"].dt.month

df_store = df[df["Store"] == 1].copy()
df_store = df_store.sort_values("Date")

df_store = df_store[["Date", "Sales", "Promo", "DayOfWeek", "Month"]]

df_store = df_store.reset_index(drop=True)

print(df_store.head())
print(df_store.tail())
print(df_store.shape)
print(df_store.isnull().sum())

In [ ]:
split_index = int(len(df_store) * 0.8)
train = df_store.iloc[:split_index]
test = df_store.iloc[split_index:]
print(train["Date"].min(), train["Date"].max())
print(test["Date"].min(), test["Date"].max())

In [ ]:
test["Naive_Pred"] = test["Sales"].shift(7)
test_naive = test.dropna()
print(test_naive[["Date", "Sales", "Naive_Pred"]].head(10))

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(test_naive["Sales"], test_naive["Naive_Pred"])
rmse = np.sqrt(mean_squared_error(test_naive["Sales"], test_naive["Naive_Pred"]))
test_naive_nonzero = test_naive[test_naive["Sales"] != 0]
mape = np.mean(np.abs(
    (test_naive_nonzero["Sales"] - test_naive_nonzero["Naive_Pred"]) /
    test_naive_nonzero["Sales"]
)) * 100
print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)

MAE: 1204.1703296703297
RMSE: 1700.8715881396395
MAPE: 28.577105712932365


In [15]:
train["lag_7"] = train["Sales"].shift(7)
train["lag_14"] = train["Sales"].shift(14)
train_lr = train.dropna()
X_train = train_lr[["lag_7", "lag_14", "Promo", "DayOfWeek", "Month"]]
y_train = train_lr["Sales"]


In [16]:
combined = pd.concat([train, test])

combined["lag_7"] = combined["Sales"].shift(7)
combined["lag_14"] = combined["Sales"].shift(14)

test_lr = combined.loc[test.index]
test_lr = test_lr.dropna()

X_test = test_lr[["lag_7", "lag_14", "Promo", "DayOfWeek", "Month"]]
y_test = test_lr["Sales"]

print(X_test.head())
print(y_test.head())

      lag_7  lag_14  Promo  DayOfWeek  Month
760  4840.0  5255.0      0          6      1
761     0.0     0.0      0          7      2
762  4781.0  3721.0      1          1      2
763  4806.0  3680.0      1          2      2
764  4310.0  3299.0      1          3      2
760    5363
761       0
762    6038
763    4901
764    4672
Name: Sales, dtype: int64


In [17]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae = np.mean(np.abs(y_test - y_pred))
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
nonzero = y_test != 0

mape = np.mean(np.abs((y_test[nonzero] - y_pred[nonzero]) / y_test[nonzero])) * 100

print("MAE:", mae)
print("RMSE:", rmse)
print("MAPE:", mape)
print(model.coef_)

MAE: 740.9319350101023
RMSE: 1147.0435353183134
MAPE: 14.124984399836716
[4.13660071e-01 3.51459626e-01 1.45997915e+03 1.95416553e+01
 1.49869037e+01]
